# PT Create HTMLs — Fast
Run as often as needed (odds update, tips update, etc.).
Requires `PT_Vorarbeiten.ipynb` to have been run once today.

## 1. Imports

In [10]:
import os, re, json, pickle, warnings
import numpy as np
import pandas as pd
import unicodedata
import requests
from datetime import datetime, date
from scipy.stats import ttest_ind
from tqdm.notebook import tqdm
from google.colab import userdata
import pytz

## 2. Mount Drive

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Config & Load Pre-computed Data

In [12]:
BASE_HIST = "/content/drive/MyDrive/PT"       if os.path.exists("/content/drive/MyDrive/PT")       else "."
BASE_OUT  = "/content/drive/MyDrive/PT/races" if os.path.exists("/content/drive/MyDrive/PT/races") else "."
TODAY     = date.today().strftime("%Y-%m-%d")

# ── Pre-computed by Vorarbeiten ───────────────────────────────────────────────
runners          = pd.read_parquet(f"{BASE_HIST}/runners_processed.parquet")
df_with_ratings  = pd.read_parquet(f"{BASE_HIST}/df_with_ratings.parquet")

with open(f"{BASE_HIST}/notepad_flags_{TODAY}.pkl", "rb") as f:
    notepad_flags = pickle.load(f)

print(f"✅ Loaded runners_processed  ({len(runners):,} rows)")
print(f"✅ Loaded df_with_ratings    ({len(df_with_ratings):,} rows)")
print(f"✅ Loaded notepad_flags      ({len(notepad_flags)} flags)")

✅ Loaded runners_processed  (302,714 rows)
✅ Loaded df_with_ratings    (259,971 rows)
✅ Loaded notepad_flags      (76 flags)


## 4. Load Today's Data  ← reload each run

In [13]:
runners_tdy  = (pd.read_parquet(f"{BASE_HIST}/runners_tdy.parquet")
                  .drop_duplicates(subset=['horseName', 'raceId']))
webTips_tdy  = (pd.read_parquet(f"{BASE_HIST}/webTips_tdy.parquet")
                  .drop_duplicates(subset=['meetingId', 'raceId']))
odds_tdy     = pd.read_parquet(f"{BASE_HIST}/odds_tdy.parquet").drop_duplicates()
races_tdy    = pd.read_parquet(f"{BASE_HIST}/races_tdy.parquet").drop_duplicates()
today_tips   = pd.read_parquet(f"{BASE_HIST}/today_tips.parquet").drop_duplicates()

# Merge today runners with race details
runners_tdy = runners_tdy.merge(
    races_tdy[['id_race', 'going', 'distance', 'name_race', 'name_meeting', 'date']],
    left_on='raceId', right_on='id_race', how='left'
)

print(f"✅ Today's data loaded  ({len(runners_tdy)} runners, {runners_tdy['raceId'].nunique()} races)")

✅ Today's data loaded  (129 runners, 11 races)


In [14]:
# ── Going overrides ──────────────────────────────────────────────────────────
if os.path.exists(f"{BASE_HIST}/going_overrides.json"):
    with open(f"{BASE_HIST}/going_overrides.json") as f:
        overrides = json.load(f)
    override_date = overrides.get('date', '')
    if override_date == TODAY:
        for meeting, going in overrides.get('by_meeting', {}).items():
            races_tdy.loc[races_tdy['name_meeting'] == meeting, 'going'] = going
            runners_tdy.loc[runners_tdy['meetingName'] == meeting, 'going'] = going
        # for race_name, going in overrides.get('by_race_name', {}).items():
        #     races_tdy.loc[races_tdy['name_race'] == race_name, 'going'] = going
        #     runners_tdy.loc[runners_tdy['meetingName'] == meeting, 'going'] = going
        print(f"✅ Going overrides applied for {TODAY}")
    else:
        print(f"⚠️  going_overrides.json dated '{override_date}' — skipping")

# ── going_category / distance_group for today ────────────────────────────────
going_mapping = {
    "Lourd": "VERY SLOW", "Très lourd": "VERY SLOW", "Collant": "VERY SLOW",
    "Souple": "SLOW",     "Très souple": "SLOW",
    "Bon souple": "FAST", "Bon": "FAST",
    "Léger": "VERY FAST", "Bon léger": "VERY FAST", "Très léger": "VERY FAST",
    "PSF Standard": "PSF", "PSF Lente": "PSF", "PSF Rapide": "PSF",
    None: None
}

def get_distance_group(m):
    if pd.isna(m): return None
    m = int(m)
    if m <= 1000: return '0-1000'
    elif m > 3600: return '>3600'
    else:
        lower = ((m - 1) // 200) * 200 + 1
        return f'{lower}-{lower+199}'

for df in [runners_tdy, races_tdy]:
    if 'going' in df.columns:
        df['going_category'] = df['going'].map(going_mapping)
        df['distance_group'] = df['distance'].apply(get_distance_group)

# draw_unique for today
def make_draw_unique(row, meeting_col):
    parts = [row['draw'], row[meeting_col], row['distance'],
             row.get('direction'), row.get('surface')]
    return ' - '.join(str(x) for x in parts if pd.notnull(x))

merged_temp = runners_tdy.merge(
    races_tdy[['id_race', 'direction', 'rail', 'surface']],
    left_on='raceId', right_on='id_race', how='left'
)
runners_tdy['draw_unique'] = merged_temp.apply(
    make_draw_unique, axis=1, meeting_col='meetingName'
)

✅ Going overrides applied for 2026-04-22


## 5. Fetch PMU Odds  ← always fetches fresh

In [15]:
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO         = "arauch6363-crypto/pmu-tracker"

def normalize_name(name: str) -> str:
    name = unicodedata.normalize("NFD", str(name).strip())
    name = "".join(c for c in name if unicodedata.category(c) != "Mn")
    return re.sub(r"^#\d+\s+", "", name).upper().strip()

def filter_before_race(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=False)
    race_start = pd.to_datetime(df["timestamp"].dt.date.astype(str) + " " + df["heure"])
    return df[df["timestamp"] < race_start]

url  = f"https://raw.githubusercontent.com/{REPO}/main/history/odds_{TODAY}.json"
resp = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
resp.raise_for_status()
data = resp.json()

rows = [
    {"timestamp": ts, "race": rk, "hippodrome": race["hippodrome"],
     "heure": race["heure"], "horse": horse, "odds": d["odds"],
     "tendance": d["tendance"], "magnitude": d["magnitude"], "favoris": d["favoris"]}
    for ts, snapshot in data.items()
    for rk, race in snapshot.items()
    for horse, d in race["horses"].items()
]
df_odds = pd.DataFrame(rows).sort_values(["race","heure","horse","timestamp"]).reset_index(drop=True)
pmu_odds_history = filter_before_race(df_odds)
pmu_odds_history["horse"] = pmu_odds_history["horse"].apply(normalize_name)
runners_tdy["_key"] = runners_tdy["horseName"].apply(normalize_name)
pmu_odds_history = (
    pmu_odds_history
    .merge(runners_tdy[["_key","horseName"]], left_on="horse", right_on="_key", how="left")
    .drop(columns=["_key","horse"])
    .dropna(subset=["horseName"])
    .reset_index(drop=True)
)
print(f"✅ PMU odds fetched  ({len(pmu_odds_history):,} rows)")

✅ PMU odds fetched  (3,481 rows)


## 6. Load HTML Functions

In [16]:
import sys, os

_module_path = os.path.join(BASE_HIST, 'pt_html_functions.py')
print(f'Looking for module at: {_module_path}')
print(f'File exists: {os.path.exists(_module_path)}')

if not os.path.exists(_module_path):
    raise FileNotFoundError(f'pt_html_functions.py not found at {_module_path}')

# %run executes the file directly — no sys.path / module cache issues
%run {_module_path}
print(f'✅ pt_html_functions loaded from {_module_path}')

Looking for module at: /content/drive/MyDrive/PT/pt_html_functions.py
File exists: True
✅ pt_html_functions loaded from /content/drive/MyDrive/PT/pt_html_functions.py


## 7. Export HTMLs  ← the fast part

In [17]:
#Keep only rows where time is greater than current Berlin time
berlin_now = datetime.now(pytz.timezone('Europe/Berlin')).time()
current_time_str = berlin_now.strftime('%H:%M:%S')
races_tdy['time'] = races_tdy['time'].astype(str)
races_tdy_filtered = races_tdy[races_tdy['time'] > current_time_str].copy().reset_index(drop=True)
runners_tdy_filtered = runners_tdy[runners_tdy['id_race'].isin(races_tdy_filtered['id_race'])].copy().reset_index(drop=True)

In [ ]:
export_all_races_html(
    df_hist          = runners,
    df_today         = runners_tdy_filtered,
    webTips_tdy      = webTips_tdy,
    today_tips       = today_tips,
    races_tdy        = races_tdy_filtered,
    df_with_ratings  = df_with_ratings,
    notepad_flags    = notepad_flags,
    pmu_odds_history = pmu_odds_history,
    output_dir       = BASE_OUT,
)

🏇 Exporting 7 races across 2 meetings...

📍 Chantilly
  [██░░░░░░░░░░░░░░░░░░] 1/7  Prix du Champ d'Alouette
  [█████░░░░░░░░░░░░░░░] 2/7  Prix Esso
  [████████░░░░░░░░░░░░] 3/7  Prix Otto
  [███████████░░░░░░░░░] 4/7  Prix de la Biche

📍 La Teste
  [██████████████░░░░░░] 5/7  Prix des Testerines
  [█████████████████░░░] 6/7  Prix de la Société Hippique de Gabarret
  [████████████████████] 7/7  Prix de l'APPE

✅ Done — 7 files saved to /content/drive/MyDrive/PT/races


['2026-04-22__Chantilly__Prix_du_Champ_d_Alouette.html',
 '2026-04-22__Chantilly__Prix_Esso.html',
 '2026-04-22__Chantilly__Prix_Otto.html',
 '2026-04-22__Chantilly__Prix_de_la_Biche.html',
 '2026-04-22__La_Teste__Prix_des_Testerines.html',
 '2026-04-22__La_Teste__Prix_de_la_Soci_t__Hippique_de_Gabarret.html',
 '2026-04-22__La_Teste__Prix_de_l_APPE.html']